In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


In [2]:
data = np.array([
    3.56,3.56,3.56,3.56,3.56,3.56,3.56,3.56,3.56,3.56,3.56,3.56,3.56,3.56,3.56,3.56,3.56,3.56,3.56,3.56,
    0.69,0.69,0.69,0.69,0.69,0.69,0.69,0.69,0.69,0.69,0.69,0.69,0.69,0.69,0.69,0.69,0.69,0.69,0.69,0.69,
    0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,
    1.84,1.84,1.84,1.84,1.84,1.84,1.84,1.84,1.84,1.84,1.84,1.84,1.84,1.84,1.84,1.84,1.84,1.84,1.84,1.84,
    3.93,3.93,3.93,3.93,3.93,3.93,3.93,3.93,3.93,3.93,3.93,3.93,3.93,3.93,3.93,3.93,3.93,3.93,3.93,3.93,
    1.25,1.25,1.25,1.25,1.25,1.25,1.25,1.25,1.25,1.25,1.25,1.25,1.25,1.25,1.25,1.25,1.25,1.25,1.25,1.25,
    0.18,0.18,0.18,0.18,0.18,0.18,0.18,0.18,0.18,0.18,0.18,0.18,0.18,0.18,0.18,0.18,0.18,0.18,0.18,0.18,
    1.13,1.13,1.13,1.13,1.13,1.13,1.13,1.13,1.13,1.13,1.13,1.13,1.13,1.13,1.13,1.13,1.13,1.13,1.13,1.13,
    0.27,0.27,0.27,0.27,0.27,0.27,0.27,0.27,0.27,0.27,0.27,0.27,0.27,0.27,0.27,0.27,0.27,0.27,0.27,0.27,
    0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,
    0.67,0.67,0.67,0.67,0.67,0.67,0.67,0.67,0.67,0.67,0.67,0.67,0.67,0.67,0.67,0.67,0.67,0.67,0.67,0.67,
    0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,
    0.61,0.61,0.61,0.61,0.61,0.61,0.61,0.61,0.61,0.61,0.61,0.61,0.61,0.61,0.61,0.61,0.61,0.61,0.61,0.61,
    0.82,0.82,0.82,0.82,0.82,0.82,0.82,0.82,0.82,0.82,0.82,0.82,0.82,0.82,0.82,0.82,0.82,0.82,0.82,0.82,
    1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,
    0.39,0.39,0.39,0.39,0.39,0.39,0.39,0.39,0.39,0.39,0.39,0.39,0.39,0.39,0.39,0.39,0.39,0.39,0.39,0.39,
    0.11,0.11,0.11,0.11,0.11,0.11,0.11,0.11,0.11,0.11,0.11,0.11,0.11,0.11,0.11,0.11,0.11,0.11,0.11,0.11,
    1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,
    1.21,1.21,1.21,1.21,1.21,1.21,1.21,1.21,1.21,1.21,1.21,1.21,1.21,1.21,1.21,1.21,1.21,1.21,1.21,1.21,
    0.72,0.72,0.72,0.72,0.72,0.72,0.72,0.72,0.72,0.72,0.72,0.72,0.72,0.72,0.72,0.72,0.72,0.72,0.72,0.72
])


In [3]:
def block_jackknife(data, stat_func, B):
    """
    Block jackknife with B blocks
    Returns: (theta_jackknife, standard_error)
    """

    n = len(data)
    m = n // B  # block size -  zaokrąglenie do int

    if m < 1:
        return None


    # mathematica's (matches data[[1;;Quotient*B]]) -> python
    data = data[:m * B]
    
    # tak samo (matches Table[data[[...]], {i, 0, B-1}])
    blocks = data.reshape(B, m)
    
    # full estimator on data -  estymator parametru liczony na pełnych danych
    theta_hat = stat_func(data)

    # jeden blok out
    # dats = Table[stat(Flatten[data[[rands[[i]]]]]), {i, B}]
    thetas = np.array([
        stat_func(np.concatenate([blocks[:i], blocks[i+1:]]).ravel())
        for i in range(B)
    ])
    theta_bar = np.mean(thetas)

    # ---- bias (3.2.1) z jack.pdf
    bias = (B - 1) * (theta_bar - theta_hat)

    # ---- bias-corrected estimator
    theta_jack = theta_hat - bias

    # var = Variance[dats] * (B-1)^2 / B
    # Mathematica Variance = sample variance (ddof=1)
    var_dats = np.var(thetas, ddof=1)
    mse = var_dats * (B - 1)**2 / B
    se = np.sqrt(mse)
    
    return theta_hat, theta_jack, se           
# =========================================================
# STATYSTYKI
# =========================================================
def stat_mean(x):
    return np.mean(x)

def stat_variance(x):
    return np.var(x, ddof=1)


In [4]:

# =========================================================
# TEST JACKKNIFE (B=20)
# =========================================================
theta_hat_mean, theta_jack_mean, se_mean = block_jackknife(data, stat_mean, B=20)
theta_hat_var,  theta_jack_var,  se_var  = block_jackknife(data, stat_variance, B=20)

print("=== MEAN ===")
print("theta_hat:", theta_hat_mean)
print("theta_jack:", theta_jack_mean)
print("SE:", se_mean)

print("\n=== VARIANCE ===")
print("theta_hat:", theta_hat_var)
print("theta_jack:", theta_jack_var)
print("SE:", se_var)

=== MEAN ===
theta_hat: 1.0445
theta_jack: 1.0445
SE: 0.23695155511978433

=== VARIANCE ===
theta_hat: 1.0694483709273184
theta_jack: 1.1230618776492691
SE: 0.5169597756166187
